
# Hate-Speech Classification EDA

In this notebook, we have imported 3 datasets from Hugging Face and 2 locally.

Main Focus:
1. hateXplain (HF)
2. iSarcasmEval
-------------------

3. ToxiGen (HF)
4. Implicit-Hate
5. TweetEval-Irony (HF)

We added the local files to a Google Drive, can download from here to import locally https://drive.google.com/drive/folders/1xjRAOX6g6ZsIOs--Fh-JsLMWnBT65vyr?usp=sharing

The implicit-hate dataset does not have any train/test/eval splits. Will need to do that in future if wish to train with it.

The other 4 datasets are already in train/test splits.


We will focus on the main two datasets, as well as Implicit-Hate, as we intend to use it for the evaluation of hate + sarcasm text.

Links:
1. https://huggingface.co/datasets/Hate-speech-CNERG/hatexplain
2. https://github.com/hate-alert/HateXplain
3.  https://github.com/iabufarha/iSarcasmEval
4.  https://github.com/SALT-NLP/implicit-hate
5. https://huggingface.co/datasets/toxigen/toxigen-data
6. https://huggingface.co/datasets/gimmaru/tweet_eval-irony




In [ ]:
# @title Install Packages
!pip install numpy==2.0 pandas==2.2.2 scikit-learn==1.5.2 umap-learn==0.5.6 datasets==2.21.0 transformers==4.44.2 accelerate==0.34.2 emoji==2.12.1 textstat==0.7.4 ftfy==6.2.3 matplotlib==3.9.2

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.9/60.9 kB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.0/19.0 MB 54.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.9/12.9 MB 68.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.7/85.7 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.3/527.3 kB 32.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 75.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.4/324.4 kB 21.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 431.4/431.4 kB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.1/105.1 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.0/43.0 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.3/8.3 MB 67.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.6

In [ ]:
# @title Imports & Config
import os, re, json, math, random, hashlib
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from datasets import load_dataset, Dataset, DatasetDict, concatenate_datasets
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.model_selection import train_test_split
import umap

plt.rcParams["figure.figsize"] = (8,5)
RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)

# Set pandas display
pd.set_option("display.max_colwidth", 120)


In [ ]:
# @title Dataset registry and utility mappers
import os, pandas as pd
from pathlib import Path
from datasets import load_dataset, Dataset

REGISTRY = {
    # HateXplain (HF)
    "hatexplain": {"hf_id": "hatexplain", "notes": "Hate/offensive/normal + targets + rationales."},

    # ToxiGen (HF)
    "toxigen": {"hf_id": "toxigen/toxigen-data", "notes": "Adversarial implicit hate."},

    # Implicit Hate (Local)
    "implicit_hate": {
        "loader": "load_implicit_hate_fused",
        "notes": "Fused SAP_full (joined stg1+stg2+stg3+SAP text) ∪ posts-only tweets",
        "local": {
            "dir": "/content/implicit_hate",
            "stg1": "implicit_hate_v1_stg1.tsv",
            "stg2": "implicit_hate_v1_stg2.tsv",
            "stg3": "implicit_hate_v1_stg3.tsv",
            "sap_posts": "implicit_hate_v1_SAP_posts.tsv",
            "posts": [
                "implicit_hate_v1_stg1_posts.tsv",
                "implicit_hate_v1_stg2_posts.tsv",
                "implicit_hate_v1_stg3_posts.tsv",
            ],
        },
    },

    # iSarcasmEval (Local)
    "isarc": {
        "loader": "load_isarc",
        "notes": "SemEval-2022 sarcasm; resolves text and 0/1 sarcasm from multiple CSVs",
        "local": [
            "/content/isarc/english_task_a.csv",
            "/content/isarc/english_task_c.csv",
            "/content/isarc/task_A_En_test.csv",
            "/content/isarc/task_B_En_test.csv",
            "/content/isarc/task_C_En_test.csv",
            "/content/isarc/train.En.csv",
        ],
    },

    # TweetEval–Irony (HF).
    "tweet_eval_irony": {
        "hf_id": "tweet_eval",
        "subset": "irony",
        "notes": "Twitter-format irony task (train/validation/test).",
    },
}


In [ ]:
# @title Custom loaders for implicit-hate
def load_implicit_hate_fused(local_cfg, verbose=True):
    """
    Returns a HF datasets.Dataset with columns:
      ['text','coarse_class','implicit_class','extra_implicit_class','target','implied_statement','source','subset']
    where subset ∈ {'SAP_full','posts_only'}.

    Expects local_cfg like:
      {
        "dir": "/content" or "/content/implicit-hate-corpus",
        "stg1": "implicit_hate_v1_stg1.tsv",
        "stg2": "implicit_hate_v1_stg2.tsv",
        "stg3": "implicit_hate_v1_stg3.tsv",
        "sap_posts": "implicit_hate_v1_SAP_posts.tsv",
        "posts": ["implicit_hate_v1_stg1_posts.tsv", "implicit_hate_v1_stg2_posts.tsv", "implicit_hate_v1_stg3_posts.tsv"]
      }
    """
    # ---- config & helpers
    DATA_DIR = Path(local_cfg.get("dir", "/content"))

    def _read_tsv(p):
        p = Path(p)
        if not p.exists():
            raise FileNotFoundError(f"Missing file: {p}")
        return pd.read_csv(p, sep="\t")

    # ---- 1) annotations keyed by ID
    stg1 = _read_tsv(DATA_DIR / local_cfg["stg1"])              # ['ID','class']
    stg2 = _read_tsv(DATA_DIR / local_cfg["stg2"])              # ['ID','class' or 'implicit_class', 'extra_implicit_class']
    stg3 = _read_tsv(DATA_DIR / local_cfg["stg3"])              # ['ID','target','implied_statement']
    sap_posts = _read_tsv(DATA_DIR / local_cfg["sap_posts"])    # ['ID','post']

    # normalize stg2 column name -> implicit_class
    if "implicit_class" not in stg2.columns and "class" in stg2.columns:
        stg2 = stg2.rename(columns={"class": "implicit_class"})

    # ---- 2) posts-only views (no ID)
    posts_list = []
    for name in local_cfg.get("posts", []):
        p = DATA_DIR / name
        if p.exists():
            dfp = pd.read_csv(p, sep="\t")
            dfp["__source_file"] = p.name
            posts_list.append(dfp)
        else:
            if verbose:
                print(f"[warn] missing posts file: {p}")
    if posts_list:
        posts_all = pd.concat(posts_list, ignore_index=True)
    else:
        # create empty structure if none present
        posts_all = pd.DataFrame(columns=["post","class","__source_file"])

    # ---- 3) SAP_full join (IDs starting with SAP_)
    stg1["ID"] = stg1["ID"].astype(str)
    stg1_sap = stg1[stg1["ID"].str.startswith("SAP_")].copy()

    sap_full = (
        stg1_sap
        .merge(sap_posts[["ID","post"]], on="ID", how="left")
        .merge(stg2[["ID","implicit_class","extra_implicit_class"]], on="ID", how="left")
        .merge(stg3[["ID","target","implied_statement"]], on="ID", how="left")
    )
    sap_full = sap_full.rename(columns={"post":"text", "class":"coarse_class"})
    sap_full["source"] = "SAP_full"
    sap_full["subset"] = "SAP_full"

    # ---- 4) posts-only tweets (robust text/label handling)
    tweets_posts = posts_all.copy()

    # Accept either 'post' or 'text' (or rare 'tweet')
    if "text" in tweets_posts.columns:
        pass
    elif "post" in tweets_posts.columns:
        tweets_posts = tweets_posts.rename(columns={"post":"text"})
    elif "tweet" in tweets_posts.columns:
        tweets_posts = tweets_posts.rename(columns={"tweet":"text"})
    else:
        tweets_posts["text"] = pd.NA

    # Coarse class if present
    if "coarse_class" not in tweets_posts.columns:
        if "class" in tweets_posts.columns:
            tweets_posts = tweets_posts.rename(columns={"class":"coarse_class"})
        else:
            tweets_posts["coarse_class"] = pd.NA

    # IMPORTANT: override any stray columns from posts files
    # (some posts files ship 'implicit_class'/'target' columns with the string "None")
    for col in ["implicit_class","extra_implicit_class","target","implied_statement"]:
        tweets_posts[col] = pd.NA  # force NA for posts-only

    # Label source/subset
    tweets_posts["source"] = (
        tweets_posts["__source_file"] if "__source_file" in tweets_posts.columns else "posts_only"
    )
    tweets_posts["subset"] = "posts_only"

    # Drop empty text
    tweets_posts = tweets_posts[tweets_posts["text"].astype(str).str.strip().ne("")]

    # ---- 5) union to master
    common_cols = [
        "text","coarse_class","implicit_class","extra_implicit_class",
        "target","implied_statement","source","subset"
    ]
    for c in common_cols:
        if c not in sap_full.columns: sap_full[c] = pd.NA
        if c not in tweets_posts.columns: tweets_posts[c] = pd.NA

    master = pd.concat([sap_full[common_cols], tweets_posts[common_cols]], ignore_index=True)
    # Normalize stringy nulls to real NA
    NULLY = {"", "none", "None", "NA", "N/A", "NaN", "nan", "null", "Null", "NULL"}
    for col in ["implicit_class","extra_implicit_class","target","implied_statement","coarse_class","text"]:
        master[col] = master[col].apply(
            lambda x: (pd.NA if isinstance(x, str) and x.strip() in NULLY else x)
        )

    # Coverage + a small sample (only if verbose)
    if verbose:
        cov = master.assign(
            has_mech = master["implicit_class"].notna(),
            has_target = master["target"].notna(),
            subset = master["source"].eq("SAP_full").map({True:"SAP_full", False:"posts_only"})
        )
        # print("subset counts:", cov["subset"].value_counts().to_dict())
        # print("Mechanism coverage overall:", cov["has_mech"].mean())
        # print("Mechanism coverage in SAP_full:", cov.query("subset=='SAP_full'")["has_mech"].mean())
        # print("Target coverage overall:", cov["has_target"].mean())
        # print("Target coverage in SAP_full:", cov.query("subset=='SAP_full'")["has_target"].mean())
        # show a couple posts_only rows (no mechanisms)
        try:
            display("-")
            # display(cov.query("subset=='posts_only'").head(3)[["text","coarse_class","implicit_class","source","subset"]])
        except Exception:
            pass

        subset_counts = master["subset"].value_counts(dropna=False).to_dict()
        mech_cov_all = float(master["implicit_class"].notna().mean()) if len(master) else 0.0
        mech_cov_sap = float(master.loc[master["subset"].eq("SAP_full"), "implicit_class"].notna().mean()) if (master["subset"].eq("SAP_full")).any() else 0.0
        # print(f"[implicit_hate_fused] rows={len(master)} | subset_counts={subset_counts} | mech_cov_all={mech_cov_all:.3f} | mech_cov_sap={mech_cov_sap:.3f}")

    # ---- 6) Convert to HF Dataset for downstream consistency
    return Dataset.from_pandas(master, preserve_index=False)


In [ ]:
# @title Custom loaders for iSarcasmEval

def load_isarc(local_paths):
    """Return a datasets.Dataset with columns:
       ['text','sarcasm','source_file'] where sarcasm ∈ {0,1}, unlabeled rows dropped.
    """
    from pathlib import Path as _Path
    dfs = []
    for p in (local_paths if isinstance(local_paths, list) else [local_paths]):
        if os.path.exists(p):
            df = pd.read_csv(p)
            df["__file"] = _Path(p).name
            dfs.append(df)
        else:
            print(f"[warn] missing isarc file: {p}")
    if not dfs:
        raise FileNotFoundError("No iSarcasmEval CSVs found.")

    raw = pd.concat(dfs, ignore_index=True)

    def _resolve_text(r):
        for k in ["text","tweet","text_0","text_1","rephrase"]:
            v = r.get(k)
            if isinstance(v, str) and v.strip():
                return v
        return None

    def _resolve_sarcasm(r):
        v = r.get("sarcasm")
        if pd.notna(v):
            try: return int(float(v))
            except: pass
        v = r.get("human_aggregated")
        if pd.notna(v):
            try: return int(float(v) >= 0.5)
            except: pass
        v = r.get("sarcasm_ref")
        if pd.notna(v):
            try: return int(float(v) >= 0.5)
            except: pass
        return None

    resolved = pd.DataFrame({
        "text": raw.apply(_resolve_text, axis=1),
        "sarcasm": raw.apply(_resolve_sarcasm, axis=1),
        "source_file": raw["__file"],
    })
    resolved = resolved[resolved["text"].notna() & resolved["sarcasm"].notna()].reset_index(drop=True)
    return Dataset.from_pandas(resolved, preserve_index=False)


In [ ]:
# @title Registry dispatcher
def try_load_hf(hf_id, subset=None, split=None):
    if subset is None:
        return load_dataset(hf_id, split=split) if split else load_dataset(hf_id)
    else:
        return load_dataset(hf_id, subset, split=split) if split else load_dataset(hf_id, subset)

def load_registry_item(key):
    meta = REGISTRY[key]

    # 1) Custom loader
    if "loader" in meta:
        loader_fn = globals().get(meta["loader"])
        if loader_fn is None:
            raise RuntimeError(f"Loader '{meta['loader']}' not defined.")
        return loader_fn(meta.get("local"))

    # 2) Local quick path (single or list)
    local_paths = meta.get("local")
    if local_paths:
        dfs = []
        for local in (local_paths if isinstance(local_paths, list) else [local_paths]):
            if os.path.exists(local):
                print(f"[Loaded] {key} from local path: {local}")
                if local.endswith(".tsv"):
                    dfs.append(pd.read_csv(local, sep="\t"))
                elif local.endswith(".csv"):
                    dfs.append(pd.read_csv(local))
                elif local.endswith(".jsonl"):
                    dfs.append(pd.read_json(local, lines=True))
                elif local.endswith(".json"):
                    dfs.append(pd.read_json(local))
                else:
                    raise ValueError("Unsupported local file type; please convert to CSV/TSV/JSON/JSONL")
            else:
                print(f"[Warn] Local path not found for {key}: {local}")
        if dfs:
            return Dataset.from_pandas(pd.concat(dfs, ignore_index=True), preserve_index=False)

    # 3) HF exact — always pass subset if present
    if "hf_id" in meta:
        try:
            ds = try_load_hf(meta["hf_id"], subset=meta.get("subset"))
            print(f"[Loaded] {key} from HF: {meta['hf_id']}" + (f" (subset={meta.get('subset')})" if meta.get("subset") else ""))
            return ds
        except Exception as e:
            print(f"[Warn] Could not load {key} via HF id {meta['hf_id']}: {e}")

    # 4) HF candidates
    for cand in meta.get("hf_candidates", []):
        try:
            ds = try_load_hf(cand)
            print(f"[Loaded] {key} from HF candidate: {cand}")
            return ds
        except Exception as e:
            print(f"[Warn] {key} candidate {cand} failed: {e}")

    raise RuntimeError(f"Could not load {key}. Provide local paths or a custom loader in REGISTRY.")


In [ ]:
# @title Code to allow peeking at datasets
# Robust one-cell peek for HF Dataset(Dict) OR local/DataFrame-like objects
from IPython.display import display
import pandas as pd

def _peek_hf(ds, n=5):
    # DatasetDict
    try:
        for split, d in ds.items():
            cols = getattr(d, "column_names", None)
            print(f"Split: {split}; size={len(d)}; columns={cols}")
            display(d.to_pandas().head(n))
        return
    except Exception:
        pass
    # Single Dataset
    cols = getattr(ds, "column_names", None)
    print(f"Size={len(ds)}; columns={cols}")
    display(ds.to_pandas().head(n))

def _peek_local(obj, n=5):
    if hasattr(obj, "to_pandas"):
        df = obj.to_pandas()
    elif isinstance(obj, pd.DataFrame):
        df = obj
    else:
        df = pd.DataFrame(obj)
    print(f"Local-like data: shape={df.shape}; columns={list(df.columns)}")
    display(df.head(n))

def _peek(key, n=5):
    ds = load_registry_item(key)  # uses your REGISTRY/dispatcher
    print(f"\n=== {key} ===")
    try:
        import datasets as _ds
        if isinstance(ds, (_ds.Dataset, _ds.DatasetDict)):
            return _peek_hf(ds, n=n)
    except Exception:
        pass
    return _peek_local(ds, n=n)

View the data from each dataset

In [ ]:
# @title Dataset cards: exact columns + head() + short markdown blurb
from IPython.display import display, Markdown

def _detect_text_col_exact(cols, preferred):
    # return the first preferred key that actually exists (case-insensitive),
    # otherwise None (we won't "guess")
    low = {c.lower(): c for c in cols}
    for k in preferred:
        if k.lower() in low:
            return low[k.lower()]
    return None

def _add_hatexplain_preview(df):
    # HateXplain 'post' is usually a list of tokens — join for readability
    if "post" in df.columns:
        try:
            return df.assign(text_preview=df["post"].apply(
                lambda x: " ".join(x) if isinstance(x, list) else str(x)))
        except Exception:
            return df
    return df

# simple per-dataset specs for text col preference + short writeup
CARD_SPECS = {
    "HateXplain": {
        "preferred_text_cols": ["post", "text"],
        "blurb": (
            "**What it contains:** tokenized text (`post`), 3-way label (hatespeech, normal, offensive), "
            "targets, and rationales (when present).\n\n"
            "**Good for:** core hate training (hatespeech, normal, offensive), target-aware analysis, "
            "and interpretability baselines."
        ),
    },
    "Implicit Hate (fused)": {
        "preferred_text_cols": ["text", "post", "tweet"],
        "blurb": (
            "**What it contains:** text, coarse class (`explicit_hate` / `implicit_hate` / `not_hate`), "
            "mechanism (`implicit_class` like irony, threatening, incitement), and sometimes "
            "`target` + `implied_statement`.\n\n"
            "**Good for:** implicit hate evaluation and mechanism-specific checks (e.g., ‘irony’ as sarcastic hate)."
        ),
    },
    "iSarcasmEval (resolved)": {
        "preferred_text_cols": ["text", "tweet"],
        "blurb": (
            "**What it contains:** text and a binary sarcasm label (`sarcasm`/`sarcastic`), "
            "optionally subtypes (irony, satire, understatement, overstatement, rhetorical question) "
            "and rephrase pairs in other files.\n\n"
            "**Good for:** training/evaluating the sarcasm head."
        ),
    },
    "TweetEval–Irony": {
        "preferred_text_cols": ["text", "tweet"],
        "blurb": (
            "**What it contains:** tweet text and an irony label (`label`: 0=non-irony, 1=irony).\n\n"
            "**Good for:** sarcasm/irony pretraining & evaluation on social-media style text."
        ),
    },
    "ToxiGen": {
        "preferred_text_cols": ["text", "generation", "prompt", "sentence"],
        "blurb": (
            "**What it contains:** prompted/generated text with toxicity indicators (e.g., `label`, "
            "`toxicity_severity`). Columns vary slightly by split.\n\n"
            "**Good for:** robustness checks and hard negatives."
        ),
    },
}

def dataset_card(name: str, ds, sample_n: int = 5):
    """
    For a HF DatasetDict or Dataset:
      - prints exact columns & shape per split
      - shows a head() (with preview for HateXplain)
      - renders a tiny markdown blurb explaining usefulness
    """
    import datasets as _ds
    spec = CARD_SPECS.get(name, {"preferred_text_cols": ["text","tweet","post"], "blurb": ""})

    display(Markdown(f"## {name} — Dataset Card"))

    items = ds.items() if isinstance(ds, _ds.DatasetDict) else [("all", ds)]
    for split, d in items:
        cols = list(d.column_names)
        shape = (len(d), len(cols))
        text_col = _detect_text_col_exact(cols, spec["preferred_text_cols"])

        # sample to pandas for a readable head
        pdf = d.select(range(min(sample_n, len(d)))).to_pandas()
        if name == "HateXplain":
            pdf = _add_hatexplain_preview(pdf)

        print(f"\nSplit: {split} | shape={shape}")
        print("Columns:", cols)
        print("Text column:", text_col if text_col else "-")
        display(pdf.head(sample_n))

    if spec.get("blurb"):
        display(Markdown(spec["blurb"]))


In [ ]:
# @title Render cards for datasets, focused on HateXplain & iSarcasmEval
hx   = load_registry_item("hatexplain")
tei  = load_registry_item("tweet_eval_irony")
isr  = load_registry_item("isarc")
tx   = load_registry_item("toxigen")
ih   = load_registry_item("implicit_hate")

dataset_card("HateXplain", hx, sample_n=10)
dataset_card("iSarcasmEval", isr, sample_n=10)
# dataset_card("Implicit Hate (fused)", ih, sample_n=5)
# dataset_card("TweetEval–Irony", tei, sample_n=5)
# dataset_card("ToxiGen", tx, sample_n=5)


[Loaded] hatexplain from HF: hatexplain
[Loaded] tweet_eval_irony from HF: tweet_eval (subset=irony)
[Loaded] toxigen from HF: toxigen/toxigen-data


'-'

## HateXplain — Dataset Card


Split: train | shape=(15383, 4)
Columns: ['id', 'annotators', 'rationales', 'post_tokens']
Text column: -


,id,annotators,rationales,post_tokens
0,23107796_gab,"{'label': [0, 2, 2], 'annotator_id': [203, 204, 233], 'target': [['Hindu', 'Islam'], ['Hindu', 'Islam'], ['Hindu', '...","[[0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0...","[u, really, think, i, would, not, have, been, raped, by, feral, hindu, or, muslim, back, in, india, or, bangladesh, ..."
1,9995600_gab,"{'label': [2, 2, 0], 'annotator_id': [27, 6, 4], 'target': [['Minority', 'Refugee', 'Indian'], ['Indian'], ['Refugee...","[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0], [0, 0, 0, 0, 0, 0...","[the, uk, has, threatened, to, return, radioactive, waste, to, the, eu, if, an, agreement, cannot, be, reached, can,..."
2,1227920812235051008_twitter,"{'label': [2, 2, 2], 'annotator_id': [209, 203, 222], 'target': [['Other'], ['Hindu'], ['Hindu', 'Other']]}","[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0]]","[if, english, is, not, imposition, then, hindi, is, also, not, imposition, shut, up, chutiya, retards, stophindiimpo..."
3,1204931715778543624_twitter,"{'label': [2, 2, 2], 'annotator_id': [235, 222, 209], 'target': [['Hindu'], ['Hindu'], ['Hindu']]}","[[0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1], [0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1]]","[no, liberal, congratulated, hindu, refugees, post, cab, because, they, hate, hindus]"
4,1179102559241244672_twitter,"{'label': [2, 2, 2], 'annotator_id': [51, 25, 4], 'target': [['Economic'], ['Economic'], ['Caucasian']]}","[[0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0...","[he, said, bro, even, your, texts, sound, redneck, whatever, the, fuck, that, means, 😂, 🤷, 🏻, ‍, ♂️, <user>]"
5,18790322_gab,"{'label': [2, 1, 1], 'annotator_id': [203, 202, 204], 'target': [['Hispanic'], ['Hispanic', 'Refugee'], ['Hispanic',...",[],"[i, live, and, work, with, many, legal, mexican, immigrants, who, are, great, citizens, and, trump, supporters, they..."
6,13459307_gab,"{'label': [0, 2, 0], 'annotator_id': [67, 57, 4], 'target': [['Women', 'Disability', 'Refugee', 'Islam', 'Homosexual...","[[0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0, 1, 1], [0, 0, 0...","[in, britain, the, prime, minister, a, childless, woman, london, mayor, a, moslem, invader, in, command, of, london,..."
7,1113705567782969344_twitter,"{'label': [1, 2, 1], 'annotator_id': [207, 221, 204], 'target': [['Hindu', 'Islam'], ['Hindu', 'Islam'], ['Arab', 'H...",[],"[<user>, <user>, there, shd, be, a, film, called, no, hindus, in, kashmir, wud, be, perfect, islamic, fanaticsm, mov..."
8,24555628_gab,"{'label': [2, 1, 1], 'annotator_id': [246, 221, 211], 'target': [['Christian', 'Islam', 'Jewish'], ['Christian', 'Is...",[],"[jews, defends, jews, regardless, muslims, defend, muslins, regardless, christian, european, attack, christians, chr..."
9,16382456_gab,"{'label': [0, 0, 0], 'annotator_id': [228, 220, 208], 'target': [['Refugee'], ['Refugee'], ['Refugee']]}","[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[the, non, partisan, congressional, budget, office, cbo, has, released, their, report, on, sens, lindsey, graham, an..."



Split: validation | shape=(1922, 4)
Columns: ['id', 'annotators', 'rationales', 'post_tokens']
Text column: -


... (3 outputs left out)

**What it contains:** tokenized text (`post`), 3-way label (hatespeech, normal, offensive), targets, and rationales (when present).

**Good for:** core hate training (hatespeech, normal, offensive), target-aware analysis, and interpretability baselines.

## iSarcasmEval — Dataset Card


Split: all | shape=(3867, 3)
Columns: ['text', 'sarcasm', 'source_file']
Text column: text


,text,sarcasm,source_file
0,"Size on the the Toulouse team, That pack is monstrous. \r\n\r\nI can't see a Welsh region ever winning this, Money t...",0.0,english_task_a.csv
1,Pinball!,0.0,english_task_a.csv
2,So the Scottish Government want people to get their booster shots so badly that the website doesn't even work,1.0,english_task_a.csv
3,villainous pro tip : change the device name on her bluetooth devices so she doesn’t forget u,0.0,english_task_a.csv
4,I would date any of these men 🥺,0.0,english_task_a.csv
5,"If there's one fictional place I could go, Christmas at Hogwarts would be that place",0.0,english_task_a.csv
6,"I mean, it's a great cause they're doing it for, but can we not have new Xmas music that's... well... good?!",0.0,english_task_a.csv
7,Where did they get 1.22xg from without the pen?!,0.0,english_task_a.csv
8,"There was so much interest in your Venice series Peg, the hand colouring is exquisite!👌🏽👏🏽👍🏽",0.0,english_task_a.csv
9,"Sometimes I lay in bed and think about how today will be the day I make my life better. Exercise, drinking water, ea...",0.0,english_task_a.csv



HateXplain
  - text: Text of the post.
  - label: 0/1/2 mapping to hatespeech, normal, offensive.
  - rationales: Token indices marked by annotators.
  - target: Targeted group or entity when present.

iSarcasmEval
  - text: Tweet or text.
  - sarcastic/sarcasm: Binary sarcasm label (1/0).
  - irony/satire/…: Sarcasm subtypes (0/1) when available.
  - rephrase/text_0/text_1: Original ↔ literal rewrite pairs (Task C).

Implicit Hate (fused)
  - text: Text (SAP_full from IDs or posts-only).
  - coarse_class: explicit_hate / implicit_hate / not_hate.
  - implicit_class: Mechanism e.g., irony, inferiority, threatening, incitement.
  - target: Free-text target (Only in Stage-3; often missing outside SAP overlap).
  - implied_statement: Free-text explanation (Only in Stage-3).
  - subset: SAP_full (has mechanisms) or posts_only (text-only).


TweetEval–Irony
  - text: Tweet text.
  - label: 0=non-irony, 1=irony.

ToxiGen
  - prompt/text: Prompt or generated text.
  - toxicity_label/toxicity_severity: Binary/graded toxicity indicators (varies by split).

In [ ]:
# @title HateXplain:deep peek helpers
import random, html
import pandas as pd
from IPython.display import display, HTML

# IMPORTANT: align to your mapping
# index -> name: 0=hate, 1=normal, 2=offensive
HX_CLASSES = ["hate", "normal", "offensive"]
VOTE_HEADER = "[hate, normal, offensive]"

def _as_list(x):
    if x is None: return []
    if isinstance(x, list): return x
    # numpy/tuples/series/etc.
    try:
        if isinstance(x, (str, bytes)): return [x]
        return list(x)
    except Exception:
        return [x]

def decode_votes_from_annotators(ann_obj):
    """
    ann_obj example:
      {'label': [1,1,1], 'annotator_id': [83,3,51], 'target': [['None'], ['None'], ['None']]}
    Returns:
      votes: [cnt_hate, cnt_normal, cnt_offensive]
      majority_idx: 0/1/2
      majority_name: str
      majority_onehot: [0/1, 0/1, 0/1] (same order as HX_CLASSES)
      annotator_count: int
      targets_flat: flattened list of targets (excludes 'None')
    """
    labels = _as_list((ann_obj or {}).get("label"))
    # ensure ints
    try:
        labels = [int(x) for x in labels]
    except Exception:
        labels = []

    votes = [0, 0, 0]  # [hate, normal, offensive]
    for vi in labels:
        if 0 <= vi <= 2:
            votes[vi] += 1

    majority_idx = max(range(3), key=lambda i: votes[i]) if sum(votes) > 0 else 1  # default to 'normal' if empty
    majority_name = HX_CLASSES[majority_idx]
    majority_onehot = [1 if i == majority_idx else 0 for i in range(3)]
    annotator_count = len(labels)

    # Flatten targets: list of lists like [['None'], ['women'], ...]
    targets = []
    raw_targets = _as_list((ann_obj or {}).get("target"))
    for t in raw_targets:
        for ti in _as_list(t):
            if isinstance(ti, str) and ti.strip() and ti.strip().lower() != "none":
                targets.append(ti.strip())
    # de-duplicate, keep order
    seen = set(); targets_flat = []
    for t in targets:
        if t not in seen:
            seen.add(t); targets_flat.append(t)

    return votes, majority_idx, majority_name, majority_onehot, annotator_count, targets_flat

def union_rationale_indices(rationales):
    """
    Accepts:
      - [] (empty)
      - [ [idxs...], [idxs...] ] per annotator
      - { 'r0':[...], 'r1':[...], ... }  (rare)
    Returns a set of token indices to highlight.
    """
    idxs = set()
    if rationales is None: return idxs
    if isinstance(rationales, dict):
        for v in rationales.values():
            for i in _as_list(v):
                try: idxs.add(int(i))
                except: pass
        return idxs
    # list or list-of-lists
    rlist = _as_list(rationales)
    if not rlist: return idxs
    if any(isinstance(x, list) for x in rlist):
        for lst in rlist:
            for i in _as_list(lst):
                try: idxs.add(int(i))
                except: pass
    else:
        for i in rlist:
            try: idxs.add(int(i))
            except: pass
    return idxs

def join_tokens(tok_list):
    return " ".join(tok_list) if isinstance(tok_list, list) else str(tok_list)

def render_text_with_rationales(tokens, idxs):
    """Return HTML with <mark> around rationale tokens."""
    pieces = []
    seq = tokens if isinstance(tokens, list) else [str(tokens)]
    for i, tok in enumerate(seq):
        esc = html.escape(tok)
        pieces.append(f"<mark>{esc}</mark>" if i in idxs else esc)
    html_block = "<div style='line-height:1.8; font-family: ui-sans-serif, system-ui;'>" + " ".join(pieces) + "</div>"
    return HTML(html_block)

def normalize_row(row, token_col="post_tokens"):
    """
    Convert a HateXplain row to a tidy dict with:
      id, text, votes, majority_name, majority_onehot, annotator_count, targets, rationale_indices
    """
    ann_obj = row.get("annotators", {})
    votes, maj_idx, maj_name, maj_onehot, ann_cnt, targets = decode_votes_from_annotators(ann_obj)
    idxs = union_rationale_indices(row.get("rationales", []))
    toks = row.get(token_col, row.get("post", row.get("text", [])))
    text = join_tokens(toks)
    return {
        "id": row.get("id", None),
        "text": text,
        "votes": votes,                         # [hate, normal, offensive]
        "majority_label": maj_name,             # 'hate' / 'normal' / 'offensive'
        "majority_onehot": maj_onehot,          # [0/1, 0/1, 0/1]
        "annotator_count": ann_cnt,
        "targets": targets,                     # flattened, deduped
        "rationale_indices": sorted(list(idxs)),
        "token_list": toks if isinstance(toks, list) else [str(toks)],
    }


In [ ]:
# @title HateXplain: 5 readable examples + summary table
import random
from IPython.display import display

HX_SPLIT = "train"     # 'train' / 'validation' / 'test'
N_EXAMPLES = 5
SEED = 42

hx = load_registry_item("hatexplain")
ds = hx[HX_SPLIT].to_pandas()

# Use your observed token column
TOKEN_COL = "post_tokens" if "post_tokens" in ds.columns else ("post" if "post" in ds.columns else "text")

rng = random.Random(SEED)
idxs = rng.sample(range(len(ds)), k=min(N_EXAMPLES, len(ds)))

tidy_rows = []
for ridx in idxs:
    row = ds.iloc[ridx].to_dict()
    nr = normalize_row(row, token_col=TOKEN_COL)
    tidy_rows.append(nr)

    print(f"\n— ID: {nr['id']}")
    print(f"  Annotators: {nr['annotator_count']} | Votes {VOTE_HEADER}: {nr['votes']} | Majority: {nr['majority_label']} | OneHot (same order): {nr['majority_onehot']}")
    print(f"  Targets: {', '.join(nr['targets']) if nr['targets'] else '(none)'}")

    if nr["rationale_indices"]:
        display(render_text_with_rationales(nr["token_list"], set(nr["rationale_indices"])))
    else:
        print("  Text:")
        print(nr["text"])

# Compact table for quick scanning
tbl = pd.DataFrame([{
    "id": r["id"],
    "majority_label": r["majority_label"],
    "votes[hate,normal,offensive]": r["votes"],
    "annotator_count": r["annotator_count"],
    "targets": ", ".join(r["targets"]) if r["targets"] else "",
    "has_rationales": bool(r["rationale_indices"]),
    "rationale_len": len(r["rationale_indices"]),
    "text_preview": (r["text"][:120] + "…") if len(r["text"]) > 120 else r["text"],
} for r in tidy_rows])

print("\nSummary (sampled rows):")
display(tbl)


[Loaded] hatexplain from HF: hatexplain

— ID: 17048275_gab
  Annotators: 3 | Votes [hate, normal, offensive]: [3, 0, 0] | Majority: hate | OneHot (same order): [1, 0, 0]
  Targets: African, Jewish
  Text:
['check' 'the' 'fucking' 'cianiggers' 'and' 'more' 'kike' 'wars' 'in'
 'this' 'fucking' 'thread' 'this' 'is' 'the' 'worst' 'propaganda' 'i'
 'have' 'ever' 'seen']

— ID: 14104384_gab
  Annotators: 3 | Votes [hate, normal, offensive]: [2, 0, 1] | Majority: hate | OneHot (same order): [1, 0, 0]
  Targets: Caucasian, African
  Text:
['why' 'does' 'anyone' 'care' 'what' 'the' 'albino' 'nigger' 'has' 'to'
 'say']

— ID: 1178538620573097985_twitter
  Annotators: 3 | Votes [hate, normal, offensive]: [1, 0, 2] | Majority: offensive | OneHot (same order): [0, 0, 1]
  Targets: Homosexual, Men
  Text:
['<user>' 'yuh' 'we' 'nigga' 'ima' 'cowboys' 'fan' 'unlike' 'ur' 'patriot'
 'faggot' 'ass']

— ID: 12852525_gab
  Annotators: 3 | Votes [hate, normal, offensive]: [1, 0, 2] | Majority: offensive |

,id,majority_label,"votes[hate,normal,offensive]",annotator_count,targets,has_rationales,rationale_len,text_preview
0,17048275_gab,hate,"[3, 0, 0]",3,"African, Jewish",False,0,['check' 'the' 'fucking' 'cianiggers' 'and' 'more' 'kike' 'wars' 'in'\n 'this' 'fucking' 'thread' 'this' 'is' 'the' ...
1,14104384_gab,hate,"[2, 0, 1]",3,"Caucasian, African",False,0,['why' 'does' 'anyone' 'care' 'what' 'the' 'albino' 'nigger' 'has' 'to'\n 'say']
2,1178538620573097985_twitter,offensive,"[1, 0, 2]",3,"Homosexual, Men",False,0,['<user>' 'yuh' 'we' 'nigga' 'ima' 'cowboys' 'fan' 'unlike' 'ur' 'patriot'\n 'faggot' 'ass']
3,12852525_gab,offensive,"[1, 0, 2]",3,Caucasian,False,0,['now' 'theres' 'the' 'inner' 'nazi' 'comming' 'out' 'just' 'like' 'most'\n 'white' 'people' 'blacklivesmatter']
4,21048577_gab,hate,"[2, 1, 0]",3,"Jewish, Refugee",False,0,['net' 'is' 'in' 'a' 'fight' 'with' 'american' 'jews' 'he' 'not' 'into'\n 'the' 'immigrant' 'agenda' 'so' 'they' 'ar...


In [ ]:
# @title iSarcasmEval helpers
import os, pandas as pd
from IPython.display import display, HTML, Markdown
from difflib import ndiff

ISARC_FILES = {
    "task_a_train_primary": "/content/isarc/english_task_a.csv",
    "task_a_train_alt":     "/content/isarc/train.En.csv",     # sometimes also has Task A labels
    "task_a_test":          "/content/isarc/task_A_En_test.csv",

    "task_b_test":          "/content/isarc/task_B_En_test.csv",

    "task_c_train":         "/content/isarc/english_task_c.csv",
    "task_c_test":          "/content/isarc/task_C_En_test.csv",
}

def _read_csv(path):
    if not os.path.exists(path):
        raise FileNotFoundError(f"Missing file: {path}")
    df = pd.read_csv(path)
    df.__file__ = os.path.basename(path)
    return df

def _maybe(cols, *cands):
    low = {c.lower(): c for c in cols}
    for c in cands:
        if c.lower() in low: return low[c.lower()]
    return None

def show_card(title, df, n=5):
    display(Markdown(f"### {title}"))
    print(f"shape={df.shape} | source={getattr(df, '__file__', '?')}")
    print("columns:", list(df.columns))
    display(df.head(n))

def diff_html(a: str, b: str) -> HTML:
    a_toks, b_toks = a.split(), b.split()
    out = []
    for tok in ndiff(a_toks, b_toks):
        if tok.startswith("- "): out.append(f"<span style='background:#ffe0e0'>{tok[2:]}</span>")
        elif tok.startswith("+ "): out.append(f"<span style='background:#e0ffe0'>{tok[2:]}</span>")
        elif tok.startswith("  "): out.append(tok[2:])
    return HTML("<div style='line-height:1.8; font-family: ui-sans-serif, system-ui;'>" + " ".join(out) + "</div>")


In [ ]:
# @title iSarcasmEval: Task A (binary sarcasm)
import numpy as np
frames = []

def pick_text_col(df):
    # Try common names, else fall back to the first object/string-like column
    for cand in ["text", "tweet", "text_0", "rephrase", "text_1"]:
        if cand in df.columns:
            return cand
        # also case-insensitive match
        low = {c.lower(): c for c in df.columns}
        if cand.lower() in low:
            return low[cand.lower()]
    # fallback: first column with dtype object
    obj_cols = [c for c in df.columns if df[c].dtype == "O"]
    return obj_cols[0] if obj_cols else df.columns[0]

def pick_label_col(df):
    for cand in ["sarcastic", "sarcasm", "label"]:
        if cand in df.columns:
            return cand
        low = {c.lower(): c for c in df.columns}
        if cand.lower() in low:
            return low[cand.lower()]
    return None  # fine for unlabeled test

def to_int_or_nan(series):
    # convert safely to 0/1 ints where possible
    return pd.to_numeric(series, errors="coerce").astype("Int64")

# Primary train
dfA = _read_csv(ISARC_FILES["task_a_train_primary"])
txt = pick_text_col(dfA)
lab = pick_label_col(dfA)
tidyA1 = pd.DataFrame({
    "text": dfA[txt].astype(str),
    "sarcasm": to_int_or_nan(dfA[lab]) if lab else pd.Series([pd.NA]*len(dfA), dtype="Int64"),
    "__file__": dfA.__file__,
    "__split__": "trainA"
})
frames.append(tidyA1)
show_card(f"Task A (train) — {dfA.__file__}", dfA)

# Optional alt train (train.En.csv)
try:
    dfA2 = _read_csv(ISARC_FILES["task_a_train_alt"])
    txt2 = pick_text_col(dfA2)
    lab2 = pick_label_col(dfA2)
    if txt2:
        tidyA2 = pd.DataFrame({
            "text": dfA2[txt2].astype(str),
            "sarcasm": to_int_or_nan(dfA2[lab2]) if lab2 else pd.Series([pd.NA]*len(dfA2), dtype="Int64"),
            "__file__": dfA2.__file__,
            "__split__": "trainA_alt"
        })
        frames.append(tidyA2)
        show_card(f"Task A (train alt) — {dfA2.__file__}", dfA2)
except FileNotFoundError:
    pass

# Test A (may be unlabeled)
dfA_test = _read_csv(ISARC_FILES["task_a_test"])
txtT = pick_text_col(dfA_test)
labT = pick_label_col(dfA_test)
tidyAT = pd.DataFrame({
    "text": dfA_test[txtT].astype(str),
    "sarcasm": to_int_or_nan(dfA_test[labT]) if labT else pd.Series([pd.NA]*len(dfA_test), dtype="Int64"),
    "__file__": dfA_test.__file__,
    "__split__": "testA"
})
frames.append(tidyAT)
show_card(f"Task A (test) — {dfA_test.__file__}", dfA_test)

taskA = pd.concat(frames, ignore_index=True)
print("\nTask A — combined tidy frame:", taskA.shape)
if taskA["sarcasm"].notna().any():
    print("Label distribution (dropna):")
    display(taskA["sarcasm"].dropna().value_counts())
else:
    print("No Task A labels found in provided files.")

### Task A (train) — english_task_a.csv

shape=(1400, 4) | source=english_task_a.csv
columns: ['text', 'sarcasm_ref', 'human_aggregated', '#humans_sarcasm']


,text,sarcasm_ref,human_aggregated,#humans_sarcasm
0,"Size on the the Toulouse team, That pack is monstrous. \r\n\r\nI can't see a Welsh region ever winning this, Money t...",0,0,1
1,Pinball!,0,0,0
2,So the Scottish Government want people to get their booster shots so badly that the website doesn't even work,1,1,4
3,villainous pro tip : change the device name on her bluetooth devices so she doesn’t forget u,0,0,0
4,I would date any of these men 🥺,0,0,1


### Task A (train alt) — train.En.csv

shape=(3468, 10) | source=train.En.csv
columns: ['Unnamed: 0', 'tweet', 'sarcastic', 'rephrase', 'sarcasm', 'irony', 'satire', 'understatement', 'overstatement', 'rhetorical_question']


,Unnamed: 0,tweet,sarcastic,rephrase,sarcasm,irony,satire,understatement,overstatement,rhetorical_question
0,0,The only thing I got from college is a caffeine addiction,1,"College is really difficult, expensive, tiring, and I often question if a degree is worth the stress.",0.0,1.0,0.0,0.0,0.0,0.0
1,1,I love it when professors draw a big question mark next to my answer on an exam because I’m always like yeah I don’t...,1,I do not like when professors don’t write out specific notes on my answers to exams.,1.0,0.0,0.0,0.0,0.0,0.0
2,2,Remember the hundred emails from companies when Covid started getting real? I’ve gotten three in regards to support ...,1,"I, at the bare minimum, wish companies actually cared about marginalized communities. At the max, that our capitalis...",0.0,1.0,0.0,0.0,0.0,0.0
3,3,Today my pop-pop told me I was not “forced” to go to college 🙃 okay sure sureeee,1,"Today my pop-pop told me I was not ""forced"" to go to college. That's not true.",1.0,0.0,0.0,0.0,0.0,0.0
4,4,"@VolphanCarol @littlewhitty @mysticalmanatee I did too, and I also reported Cancun Cruz not worrying about the heart...",1,I would say Ted Cruz is an asshole and doesn’t actually care about people/babies/fetuses. He just wants to wage a wa...,1.0,0.0,0.0,0.0,0.0,0.0


### Task A (test) — task_A_En_test.csv

shape=(1400, 2) | source=task_A_En_test.csv
columns: ['text', 'sarcastic']


,text,sarcastic
0,"Size on the the Toulouse team, That pack is monstrous. \n\nI can't see a Welsh region ever winning this, Money talks...",0
1,Pinball!,0
2,So the Scottish Government want people to get their booster shots so badly that the website doesn't even work,1
3,villainous pro tip : change the device name on her bluetooth devices so she doesn’t forget u,0
4,I would date any of these men 🥺,0



Task A — combined tidy frame: (6268, 4)
Label distribution (dropna):


,count
sarcasm,
0,3801
1,1067


In [ ]:
# @title
if taskA["sarcasm"].notna().any():
  print("\nTask A — sample sarcastic (label=1)")
  display(taskA[taskA["sarcasm"] == 1][["text","sarcasm","__file__","__split__"]].head(3))
  print("\nTask A — sample non-sarcastic (label=0)")
  display(taskA[taskA["sarcasm"] == 0][["text","sarcasm","__file__","__split__"]].head(3))


Task A — sample sarcastic (label=1)


,text,sarcasm,__file__,__split__
1400,The only thing I got from college is a caffeine addiction,1,train.En.csv,trainA_alt
1401,I love it when professors draw a big question mark next to my answer on an exam because I’m always like yeah I don’t...,1,train.En.csv,trainA_alt
1402,Remember the hundred emails from companies when Covid started getting real? I’ve gotten three in regards to support ...,1,train.En.csv,trainA_alt



Task A — sample non-sarcastic (label=0)


,text,sarcasm,__file__,__split__
2267,I always think going braless is a good idea until I'm in public and am insecure because I'm not wearing a bra,0,train.En.csv,trainA_alt
2268,life is so much better with a heating blanket,0,train.En.csv,trainA_alt
2269,Sometimes I just go through my phone and look at pictures of my dog,0,train.En.csv,trainA_alt


In [ ]:
# @title iSarcasmEval: Task B (subtypes) — tidy view + prevalence
dfB = _read_csv(ISARC_FILES["task_b_test"])
txt = _maybe(dfB.columns, "text", "tweet")
sub_cols = [c for c in dfB.columns if c.lower() in
            {"sarcasm","irony","satire","understatement","overstatement","rhetorical_question"}]
taskB = dfB[[txt] + sub_cols].rename(columns={txt:"text"}) if txt else dfB[sub_cols].copy()
taskB["__file__"] = dfB.__file__; taskB["__split__"] = "testB"

show_card("Task B (subtypes) — from task_B_En_test.csv", dfB)
print("\nSubtype prevalence (mean of binary columns):")
display(taskB.drop(columns=["__file__","__split__","text"], errors="ignore").mean().sort_values(ascending=False))


### Task B (subtypes) — from task_B_En_test.csv

shape=(1400, 7) | source=task_B_En_test.csv
columns: ['text', 'sarcasm', 'irony', 'satire', 'understatement', 'overstatement', 'rhetorical_question']


,text,sarcasm,irony,satire,understatement,overstatement,rhetorical_question
0,saw a video of someone getting a hug. would LOVE a hug. u know those really nice ones,0,0,0,0,0,0
1,"""This Christmas I hope you all either get vaccinated or wear a mask. The omicron variant is not going to give us a n...",0,0,0,0,0,0
2,It's the alamo,0,0,0,0,0,0
3,"Wind 5 mph E. Barometer 1029.8 hPa, Pressure trend. Temperature 8.1 °C. Rain today 2.1mm. Humidity 96%",0,0,0,0,0,0
4,I constantly have loads of the new symptoms but also negative tests. Tsk about making you paranoid.,0,0,0,0,0,0



Subtype prevalence (mean of binary columns):


,0
sarcasm,0.128571
satire,0.035000
irony,0.014286
rhetorical_question,0.007857
overstatement,0.007143
understatement,0.000714


In [ ]:
# @title
# Show a few positives per subtype so it's obvious labels exist
if not taskB.empty:
    label_cols = [c for c in taskB.columns if c.lower() in
                  {"sarcasm","irony","satire","understatement","overstatement","rhetorical_question"}]
    for col in label_cols:
        pos = taskB[taskB[col] == 1]
        print(f"\nExamples with {col}=1 (showing up to 3): {len(pos)} found")
        display(pos[["text", col]].head(3))



Examples with sarcasm=1 (showing up to 3): 180 found


,text,sarcasm
10,"Ah, we really needed that vaccine booster address from Boris Johnson, and it wasn't ANYTHING to do with getting the ...",1
12,"Because Karen from YouTube knows more about science than, you know, actual scientists",1
15,Love it when I get disturbed three times in the bath to be asked where things in the house are!!,1



Examples with irony=1 (showing up to 3): 20 found


,text,irony
37,Oh what a surprise BBC weather says it's raining in Manchester! in Manchester!,1
88,Weathers wonderful today! 🌂🌂🌂,1
203,Everybody loves Christmas! It such a great time to be jolly!,1



Examples with satire=1 (showing up to 3): 49 found


,text,satire
16,"How kind of the Conservative party to cancel their Christmas get together this year, I mean they didn't have enough ...",1
88,Weathers wonderful today! 🌂🌂🌂,1
118,"Well, at least the covid situation seems to be improving in time for christmas...",1


... (2 outputs left out)


Examples with overstatement=1 (showing up to 3): 10 found


,text,overstatement
506,You on your best behaviour is me on my worst,1
533,"Omicron: Cases are going up, run for the hills.",1
553,So kind of the government to introduce 900 new covid tests right before I fly to Paris x,1



Examples with rhetorical_question=1 (showing up to 3): 11 found


,text,rhetorical_question
16,"How kind of the Conservative party to cancel their Christmas get together this year, I mean they didn't have enough ...",1
64,Sure you do! Doesn't everyone?! I do it everyday!!!,1
82,"When someone puts on too much perfume I would say nice perfume, how long did you let it marinate???Tt",1


In [ ]:
# @title iSarcasmEval: Task C (pairs) — tidy view + 3 diffs
pairs = []

# Train pairs
dfC_tr = _read_csv(ISARC_FILES["task_c_train"])
t0 = _maybe(dfC_tr.columns, "text_0", "tweet", "text")
t1 = _maybe(dfC_tr.columns, "text_1", "rephrase")
pairs.append(pd.DataFrame({"text_orig": dfC_tr[t0].astype(str), "text_rephrase": dfC_tr[t1].astype(str), "__file__": dfC_tr.__file__, "__split__": "trainC"}))
show_card("Task C (train) — from english_task_c.csv", dfC_tr)

# Test pairs
dfC_te = _read_csv(ISARC_FILES["task_c_test"])
t0e = _maybe(dfC_te.columns, "text_0", "tweet", "text")
t1e = _maybe(dfC_te.columns, "text_1", "rephrase")
pairs.append(pd.DataFrame({"text_orig": dfC_te[t0e].astype(str), "text_rephrase": dfC_te[t1e].astype(str), "__file__": dfC_te.__file__, "__split__": "testC"}))
show_card("Task C (test) — from task_C_En_test.csv", dfC_te)

taskC = pd.concat(pairs, ignore_index=True)
print("\nTask C — combined tidy frame:", taskC.shape)


### Task C (train) — from english_task_c.csv

shape=(200, 5) | source=english_task_c.csv
columns: ['text_0', 'text_1', 'sarcastic_id_ref', 'human_aggregated', 'human_votes']


,text_0,text_1,sarcastic_id_ref,human_aggregated,human_votes
0,I see that your team played well today!,I'm sorry that your team didn't win yesterday.,0,0,5
1,"Anthony Taylor is such a fair referee, I wish he was put in charge of more chelsea matches",I hope Anthony Taylor is never put in charge of a Chelsea match again,0,0,5
2,"the weather is gloomy, just raining and dull.",What a glorious weather today,1,1,5
3,People going out to get there boosters without thinking if they a) need them b) offer a reasonable level of protecti...,Nice to see the sheep getting their boosters to fully protect themselves,1,1,5
4,"Really great weather we're having, love a bit of January heat and sunshine. Almost as good as Newcastle's form...",Really cold January so far - looking forward to some warmer and brighter days ahead! #notafanofwinter,0,0,5


### Task C (test) — from task_C_En_test.csv

shape=(200, 3) | source=task_C_En_test.csv
columns: ['text_0', 'text_1', 'sarcastic_id']


,text_0,text_1,sarcastic_id
0,I see that your team played well today!,I'm sorry that your team didn't win yesterday.,0
1,"Anthony Taylor is such a fair referee, I wish he was put in charge of more chelsea matches",I hope Anthony Taylor is never put in charge of a Chelsea match again,0
2,"the weather is gloomy, just raining and dull.",What a glorious weather today,1
3,People going out to get there boosters without thinking if they a) need them b) offer a reasonable level of protecti...,Nice to see the sheep getting their boosters to fully protect themselves,1
4,"Really great weather we're having, love a bit of January heat and sunshine. Almost as good as Newcastle's form...",Really cold January so far - looking forward to some warmer and brighter days ahead! #notafanofwinter,0



Task C — combined tidy frame: (400, 4)


In [ ]:
# @title Implicit Hate: load raw files (stg1/stg2/stg3 + SAP text + posts-only)
from pathlib import Path
import pandas as pd

IH_CFG = REGISTRY["implicit_hate"]["local"]  # uses your existing registry
DATA_DIR = Path(IH_CFG.get("dir", "/content"))

def _read_tsv(p):
    p = Path(p)
    if not p.exists():
        raise FileNotFoundError(f"Missing file: {p}")
    return pd.read_csv(p, sep="\t")

# Stage files keyed by ID
ih_stg1 = _read_tsv(DATA_DIR / IH_CFG["stg1"])          # ['ID','class']
ih_stg2 = _read_tsv(DATA_DIR / IH_CFG["stg2"])          # ['ID','class'(=implicit_class), 'extra_implicit_class']
ih_stg3 = _read_tsv(DATA_DIR / IH_CFG["stg3"])          # ['ID','target','implied_statement']
sap_posts = _read_tsv(DATA_DIR / IH_CFG["sap_posts"])   # ['ID','post']

# Posts-only views (no IDs)
posts_only_files = [DATA_DIR / p for p in IH_CFG.get("posts", [])]
posts_only_list = []
for p in posts_only_files:
    if p.exists():
        dfp = pd.read_csv(p, sep="\t")
        dfp["__source_file"] = p.name
        posts_only_list.append(dfp)
posts_only = pd.concat(posts_only_list, ignore_index=True) if posts_only_list else pd.DataFrame()

print("stg1:", ih_stg1.shape, list(ih_stg1.columns))
print("stg2:", ih_stg2.shape, list(ih_stg2.columns))
print("stg3:", ih_stg3.shape, list(ih_stg3.columns))
print("sap_posts:", sap_posts.shape, list(sap_posts.columns))
print("posts_only:", posts_only.shape, list(posts_only.columns))


stg1: (21480, 2) ['ID', 'class']
stg2: (6346, 3) ['ID', 'implicit_class', 'extra_implicit_class']
stg3: (6359, 3) ['ID', 'target', 'implied_statement']
sap_posts: (548, 2) ['ID', 'post']
posts_only: (34185, 7) ['post', 'class', '__source_file', 'implicit_class', 'extra_implicit_class', 'target', 'implied_statement']


In [ ]:
# @title Implicit Hate: Stage-by-stage EDA cards
from IPython.display import display, Markdown

# Normalize stg2 column name
if "implicit_class" not in ih_stg2.columns and "class" in ih_stg2.columns:
    ih_stg2 = ih_stg2.rename(columns={"class": "implicit_class"})

display(Markdown("## Stage 1 — Coarse classes"))
display(ih_stg1.head(5))
print("\nClass distribution (stg1):")
display(ih_stg1["class"].value_counts(dropna=False))

display(Markdown("## Stage 2 — Implicit mechanisms"))
display(ih_stg2.head(5))
print("\nMechanism distribution (stg2):")
mech_counts = ih_stg2["implicit_class"].astype(str).str.lower().value_counts()
display(mech_counts)

if "extra_implicit_class" in ih_stg2.columns:
    extra_counts = ih_stg2["extra_implicit_class"].astype(str).str.lower().value_counts()
    print("\nExtra mechanism (secondary) distribution:")
    display(extra_counts)

# Combined primary+extra mechanism histogram
stacked = pd.concat([
    ih_stg2["implicit_class"].astype(str).str.lower(),
    ih_stg2["extra_implicit_class"].astype(str).str.lower()
], ignore_index=True)
stacked = stacked[stacked.notna() & (stacked != "nan")]
print("\nCombined mechanism counts (primary+extra):")
display(stacked.value_counts())

display(Markdown("## Stage 3 — Targets & implied statements (explanations)"))
display(ih_stg3.head(5))
print("\nTop targets (stg3):")
display(ih_stg3["target"].dropna().astype(str).value_counts().head(20))
print("\nExample implied statements:")
display(ih_stg3[["implied_statement"]].dropna().head(5))


## Stage 1 — Coarse classes

,ID,class
0,399886440588247041,implicit_hate
1,929901925100937216,not_hate
2,728678509497954304,not_hate
3,625688620444180481,not_hate
4,441089979322597376,not_hate



Class distribution (stg1):


,count
class,
not_hate,13291
implicit_hate,7100
explicit_hate,1089


## Stage 2 — Implicit mechanisms

,ID,implicit_class,extra_implicit_class
0,399886440588247041,white_grievance,NaN
1,755109998770921473,irony,NaN
2,251422864232808449,white_grievance,NaN
3,548214253497249792,stereotypical,NaN
4,184109665108504579,stereotypical,NaN


... (8 outputs left out)


Top targets (stg3):


,count
target,
Minorities,732
Immigrants,653
white people,618
Muslims,574
Jews,436
non-white people,253
Blacks,249
immigrants,161
Black people,154



Example implied statements:


,implied_statement
0,people in minority groups are all in gangs
1,non-white people are dangerous
2,Immigrants are undesirable
3,Anti-whites hate white people
4,Capitalists don't like white people.


In [ ]:
# @title Implicit Hate: SAP_full join preview (IDs + text + mechanisms + targets)
# Keep only SAP_* IDs from stg1
ih_stg1_sap = ih_stg1[ih_stg1["ID"].astype(str).str.startswith("SAP_")].copy()

sap_join = (ih_stg1_sap
            .merge(sap_posts[["ID","post"]], on="ID", how="left")
            .merge(ih_stg2.rename(columns={"class":"implicit_class"}) if "class" in ih_stg2.columns else ih_stg2,
                   on="ID", how="left")
            .merge(ih_stg3, on="ID", how="left"))

sap_join = sap_join.rename(columns={"class":"coarse_class", "post":"text"})
print("SAP_full preview shape:", sap_join.shape)

# Coverage checks
mech_cov = sap_join["implicit_class"].notna().mean()
tgt_cov  = sap_join["target"].notna().mean()
print(f"Mechanism coverage in SAP_full: {mech_cov:.3f} | Target coverage: {tgt_cov:.3f}")

display(sap_join.head(8)[["ID","text","coarse_class","implicit_class","extra_implicit_class","target","implied_statement"]])

# Show a few "hateful + sarcastic" (irony mechanism) examples
irony_rows = sap_join[
    (sap_join["coarse_class"].astype(str).str.lower()=="implicit_hate") &
    (sap_join["implicit_class"].astype(str).str.lower()=="irony")
][["text","coarse_class","implicit_class"]].head(5)
print("\nSample implicit_hate + irony rows:")
display(irony_rows)


SAP_full preview shape: (548, 7)
Mechanism coverage in SAP_full: 0.991 | Target coverage: 0.000


,ID,text,coarse_class,implicit_class,extra_implicit_class,target,implied_statement
0,SAP_384,@NYRKelsMads Dat damn dog eats betta den most colored folk do.,implicit_hate,inferiority,NaN,NaN,NaN
1,SAP_4525,Dicks are basically ar-15s Only disabled kids whip em out in class,implicit_hate,irony,inferiority,NaN,NaN
2,SAP_4557,I have a black man in my family tree He’s still hanging there,implicit_hate,irony,inferiority,NaN,NaN
3,SAP_4563,What's faster than a school full of black kids? A loaded gun.,implicit_hate,threatening,NaN,NaN,NaN
4,SAP_4569,What did the jew say to the other jew? Jew: Is something burning?,implicit_hate,irony,NaN,NaN,NaN
5,SAP_4573,"Why did so many blacks die in combat during Vietnam? Every time the Sergeant yelled, ""GET DOWN!"" they would get up a...",implicit_hate,irony,NaN,NaN,NaN
6,SAP_4579,What's the difference between a jew and a piece of bread? The bread still works after it's been through the toaster,implicit_hate,irony,NaN,NaN,NaN
7,SAP_4591,What two kinds of people are totally different but always look the same? Asians and hepatitis patients,implicit_hate,irony,inferiority,NaN,NaN



Sample implicit_hate + irony rows:


,text,coarse_class,implicit_class
1,Dicks are basically ar-15s Only disabled kids whip em out in class,implicit_hate,irony
2,I have a black man in my family tree He’s still hanging there,implicit_hate,irony
4,What did the jew say to the other jew? Jew: Is something burning?,implicit_hate,irony
5,"Why did so many blacks die in combat during Vietnam? Every time the Sergeant yelled, ""GET DOWN!"" they would get up a...",implicit_hate,irony
6,What's the difference between a jew and a piece of bread? The bread still works after it's been through the toaster,implicit_hate,irony


In [ ]:
# @title Build and save the 2×2 evaluation set (Hate × Sarcasm conditions)

import pandas as pd
from pathlib import Path

ih_df = load_registry_item("implicit_hate").to_pandas()
isarc_df = load_registry_item("isarc").to_pandas()

ih_sap = ih_df[ih_df["subset"] == "SAP_full"].copy()
ih_sap["coarse_class"] = ih_sap["coarse_class"].astype(str).str.lower()
ih_sap["implicit_class"] = ih_sap["implicit_class"].astype(str).str.lower()

hs_sarc     = ih_sap[(ih_sap["coarse_class"] == "implicit_hate") & (ih_sap["implicit_class"] == "irony")][["text"]]
hs_nonsarc  = pd.concat([
                    ih_sap[ih_sap["coarse_class"] == "explicit_hate"][["text"]],
                    ih_sap[(ih_sap["coarse_class"] == "implicit_hate") & (ih_sap["implicit_class"] != "irony")][["text"]],
                ], ignore_index=True)
nh_sarc     = isarc_df[isarc_df["sarcasm"] == 1][["text"]]
nh_nonsarc  = isarc_df[isarc_df["sarcasm"] == 0][["text"]]

def tag(df, hate, sarc, cell):
    out = df.copy()
    out["true_hate"] = hate
    out["sarcasm"]   = sarc
    out["cell"]      = cell
    return out

eval_df = pd.concat([
    tag(hs_sarc,     1, 1, "Hateful + Sarcastic"),
    tag(hs_nonsarc,  1, 0, "Hateful + Non-sarcastic"),
    tag(nh_sarc,     0, 1, "Not hateful + Sarcastic"),
    tag(nh_nonsarc,  0, 0, "Not hateful + Non-sarcastic"),
], ignore_index=True)

counts = eval_df.groupby(["true_hate","sarcasm"]).size().unstack().rename(columns={0:"Non-sarcastic",1:"Sarcastic"})
counts.index = counts.index.map({0:"Not hateful",1:"Hateful"})
counts["Row total"] = counts.sum(axis=1)
counts.loc["Column total"] = list(counts.sum(axis=0))
print("2×2 condition table:")
display(counts)

# Save for reuse in later training/eval
OUT = Path("/content/eda_exports")
OUT.mkdir(parents=True, exist_ok=True)
eval_df.to_csv(OUT / "eval_grid_hate_x_sarcasm.csv", index=False)
print("Saved to:", OUT / "eval_grid_hate_x_sarcasm.csv")


'-'

2×2 condition table:


sarcasm,Non-sarcastic,Sarcastic,Row total
true_hate,,,
Not hateful,2556,1311,3867
Hateful,189,359,548
Column total,2745,1670,4415


Saved to: /content/eda_exports/eval_grid_hate_x_sarcasm.csv
